In [7]:
import pandas as pd
import numpy as np
import os

import mudata as md
import anndata as ad
import seaborn as sns
from scipy.stats import norm
import matplotlib.pyplot as plt


## read AnnData and create MuData

In [ ]:
data_dir = "/data/pinello/SHARED_DATA/genome_wide_Perturb-seq/"

In [13]:
!ls -lhS {data_dir}

total 98G
-rw-r--r--. 1 ljb80 pinello  62G Apr 13 16:46 K562_gwps_raw_singlecell_01.h5ad
-rw-r--r--. 1 ljb80 pinello  10G Jun  8  2022 K562_essential_raw_singlecell_01.h5ad
-rw-r--r--. 1 ljb80 pinello 8.2G Jun  8  2022 rpe1_raw_singlecell_01.h5ad
-rw-------. 1 ljb80 pinello 113M Apr 13 04:40 nohup.out


In [14]:
%%time
adata = ad.read_h5ad(f'{data_dir}/K562_gwps_raw_singlecell_01.h5ad')
adata

AnnData object with n_obs × n_vars = 1989578 × 8248
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'

In [63]:
# sgID_A, sgID_B = zip(*adata.obs['sgID_AB'].str.split('|'))
# gene_A = [x.split('_')[0] for x in sgID_A]
# gene_B = [x.split('_')[0] for x in sgID_B]
# sgrna_df = pd.DataFrame(dict(sgID_A=sgID_A, sgID_B=sgID_B, gene_A=gene_A, gene_B=gene_B), index = adata.obs_names)
# assert(np.all(sgrna_df['gene_A']==sgrna_df['gene_B']))


In [64]:
gdata = ad.AnnData(pd.get_dummies(adata.obs['sgID_AB'], sparse=True), dtype=np.bool_)
gdata

AnnData object with n_obs × n_vars = 1989578 × 11187

In [65]:
gdata.var = gdata.var.assign(target=lambda x: [chunk[0] for chunk in x.index.str.split('_')])
# gdata.var = gdata.var.reset_index(names=['sgID_AB'])
gdata.var

,target
A1BG_+_58858964.23-P1|A1BG_-_58858788.23-P1,A1BG
A1BG_-_58864840.23-P2|A1BG_-_58864822.23-P2,A1BG
AAAS_-_53715438.23-P1P2|AAAS_+_53715355.23-P1P2,AAAS
AACS_+_125549983.23-P1P2|AACS_-_125550169.23-P1P2,AACS
AAED1_-_99417574.23-P1P2|AAED1_+_99417525.23-P1P2,AAED1
...,...
non-targeting_03756|non-targeting_01543,non-targeting
non-targeting_03758|non-targeting_00493,non-targeting
non-targeting_03778|non-targeting_00397,non-targeting
non-targeting_03783|non-targeting_02098,non-targeting


In [70]:
mdata = md.MuData({'rna':adata, 'grna':gdata})
mdata

MuData object with n_obs × n_vars = 1989578 × 19435
  2 modalities
    rna:	1989578 x 8248
      obs:	'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count'
      var:	'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    grna:	1989578 x 11187
      var:	'target'

In [86]:
mdata['rna'].var.query("gene_name=='TBCE'")
mdata['rna'].var.query("`class`=='gene_version2'")
# mdata['rna'].var.value_counts('gene_name')

gene_name
HSPA14    2
TBCE      2
PUS7L     1
PUM3      1
PURA      1
         ..
GNA13     1
GNA12     1
GNA11     1
GMPS      1
ZZEF1     1
Length: 8246, dtype: int64

In [71]:
mdata['grna'].X

<1989578x11187 sparse matrix of type '<class 'numpy.bool_'>'
	with 1989578 stored elements in Compressed Sparse Row format>

# split into chunks and write file

In [75]:
from math import ceil

n_genes = mdata['rna'].shape[1]
print(n_genes)
n_chunks = 8
chunk_size=ceil(n_genes/n_chunks) # number of genes per file chunk to yield 8 chunks

8248


In [74]:
# mdata_subset = md.MuData({, 'grna':mdata['grna']})
for i in range(0,n_chunks+1):
    # divide RNA count matrix into chunks of size chunk_size
    start = i*chunk_size
    end = min(start+chunk_size, n_genes)
    print(f"subsetting genes {start+1} to {end}")
    rna_adata_subset = mdata['rna'][:,start:end]
    mdata_subset = md.MuData({"rna":rna_adata_subset, "grna":mdata['grna']})
    print(mdata_subset)
    chunk_path = f"{data_dir}/K562_gwps_raw_singlecell.chunk_{i:02d}.h5mu"
    print(f"writing to {chunk_path}")
    mdata_subset.write(chunk_path)
# mdata['grna'].var.merge(chunk_grnas.rename(column}))

subsetting genes 1 to 2062
MuData object with n_obs × n_vars = 1989578 × 13249
  2 modalities
    rna:	1989578 x 2062
      obs:	'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count'
      var:	'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    grna:	1989578 x 11187
      var:	'target'
writing to /data/pinello/SHARED_DATA/genome_wide_Perturb-seq//K562_gwps_raw_singlecell.chunk_00.h5mu
subsetting genes 2063 to 4124
MuData object with n_obs × n_vars = 1989578 × 13249
  2 modalities
    rna:	1989578 x 2062
      obs:	'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count'
      var:	'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    grna:	1989578 x 

## Future: get position of gRNA targets

In [50]:
# gdata.var.assign(gene_name=lambda x:x['target']).merge(adata.var, how="left").set_index('sgID_AB')[['target','gene_name','chr', 'start', 'end']]